# 📄 LLM File System Assistant — Project Workbook

This notebook demonstrates a complete end-to-end implementation of an **LLM-powered File System Assistant** that uses **function calling / tool use** to interact with resume documents.

---

## 🗂️ Project Overview

| Component | Description |
|---|---|
| **Part A** | Core File System Tools (`read_file`, `list_files`, `write_file`, `search_in_file`) |
| **Part B** | LLM Integration with function calling (OpenAI / OpenRouter API) |

### Learning Objectives
- Understand LLM function calling / tool use patterns
- Implement structured tool interfaces with JSON schemas
- Handle file I/O operations programmatically (PDF, DOCX, TXT)
- Parse and validate documents

### Files in the Project
```
llm-file-system-assistant/
├── fs_tools.py             # Core file system tools (Part A)
├── llm_file_assistant.py   # LLM integration & function calling (Part B)
├── requirements.txt        # Dependencies
├── .env                    # API key configuration
├── README.md               # Project documentation
├── workbook.ipynb          # This notebook
└── resumes/                # Sample PDF resume files
    ├── resume_john_doe.pdf
    ├── resume_bob_williams.pdf
    └── resume_eva_davis.pdf
```

---
## ⚙️ Step 1 — Install Dependencies

Install all required Python packages. These are the libraries needed to parse PDF/DOCX files, communicate with the LLM API, and generate sample resumes.

In [ ]:
# Install all required dependencies
%pip install openai python-dotenv pypdf python-docx reportlab --quiet

---
## 📦 Step 2 — Import Libraries

Import all third-party and standard-library modules used throughout the project.

In [ ]:
import os
import json
import datetime
from pathlib import Path

# PDF parsing
from pypdf import PdfReader

# DOCX parsing
import docx

# PDF generation
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

# LLM API
from dotenv import load_dotenv
from openai import OpenAI

print("All libraries imported successfully.")

---
## 🔑 Step 3 — Configure API Keys

Load environment variables from the `.env` file. 
Ensure your `.env` contains one of the following:
```ini
OPENROUTER_API_KEY=sk-or-v1-...
# OR
OPENAI_API_KEY=sk-...
```

In [ ]:
load_dotenv()

OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY")
OPENAI_KEY = os.environ.get("OPENAI_API_KEY")

if OPENROUTER_KEY:
    print("Using: OpenRouter API")
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_KEY
    )
    MODEL = os.environ.get("MODEL_NAME", "openai/gpt-4o-mini")
elif OPENAI_KEY:
    print("Using: OpenAI API")
    client = OpenAI(api_key=OPENAI_KEY)
    MODEL = os.environ.get("MODEL_NAME", "gpt-4o-mini")
else:
    raise EnvironmentError("No API key found. Please set OPENROUTER_API_KEY or OPENAI_API_KEY in your .env file.")

print(f"Model: {MODEL}")

---
## 📁 Step 4 — Generate Sample PDF Resumes

Create 5 realistic dummy resume PDF files in the `resumes/` directory. These files will serve as our test dataset for all subsequent operations.

In [ ]:
def create_pdf_resume(filepath, title, summary, skills, experience, education):
    """Helper to generate a formatted PDF resume using ReportLab."""
    doc = SimpleDocTemplate(filepath, pagesize=letter)
    styles = getSampleStyleSheet()
    story = []

    title_style = ParagraphStyle('TitleStyle', parent=styles['Heading1'], fontSize=18, spaceAfter=10)
    story.append(Paragraph(title, title_style))
    story.append(Spacer(1, 10))

    story.append(Paragraph("<b>Professional Summary:</b>", styles['Heading2']))
    story.append(Paragraph(summary, styles['Normal']))
    story.append(Spacer(1, 10))

    story.append(Paragraph("<b>Technical Skills:</b>", styles['Heading2']))
    story.append(Paragraph(", ".join(skills), styles['Normal']))
    story.append(Spacer(1, 10))

    story.append(Paragraph("<b>Work Experience:</b>", styles['Heading2']))
    for exp in experience:
        story.append(Paragraph(f"• <b>{exp['role']}</b> - {exp['company']} ({exp['years']})", styles['Normal']))
        story.append(Paragraph(exp['details'], styles['Normal']))
        story.append(Spacer(1, 5))

    story.append(Paragraph("<b>Education:</b>", styles['Heading2']))
    story.append(Paragraph(education, styles['Normal']))
    doc.build(story)

RESUMES = [
    {
        "filename": "resume_john_doe.pdf",
        "title": "John Doe - Senior Software Engineer",
        "summary": "Experienced Software Engineer specializing in backend infrastructure, REST API design, and distributed systems. Expert in Python, FastAPI, and Docker.",
        "skills": ["Python", "FastAPI", "Docker", "PostgreSQL", "AWS", "Redis", "Git"],
        "experience": [
            {"role": "Senior Python Developer", "company": "TechCorp Innovations", "years": "2021 - Present",
             "details": "Architected high-throughput microservices using Python and FastAPI. Reduced API latency by 40% using Redis caching and optimized PostgreSQL queries."},
            {"role": "Software Engineer", "company": "CloudScale Systems", "years": "2018 - 2021",
             "details": "Built containerized data pipelines with Python and Docker on AWS ECS."}
        ],
        "education": "B.S. in Computer Science, University of California, Berkeley (2018)"
    },
    {
        "filename": "resume_jane_smith.pdf",
        "title": "Jane Smith - Lead Data Scientist",
        "summary": "Data Scientist with 6+ years building predictive ML models, NLP engines, and big data analytics pipelines using Python, PyTorch, and SQL.",
        "skills": ["Python", "PyTorch", "SQL", "Pandas", "Scikit-Learn", "Machine Learning", "NLP"],
        "experience": [
            {"role": "Lead Data Scientist", "company": "Analytics AI Labs", "years": "2022 - Present",
             "details": "Led a team of 4 data scientists in deploying Deep Learning models using Python and PyTorch. Improved recommendation engine precision by 25%."},
            {"role": "Data Analyst", "company": "Data Insights Co.", "years": "2019 - 2022",
             "details": "Performed customer churn analysis and ETL pipeline automation with Python, Pandas, and SQL."}
        ],
        "education": "M.S. in Statistics & Data Science, Stanford University (2019)"
    },
    {
        "filename": "resume_bob_williams.pdf",
        "title": "Bob Williams - DevOps & Cloud Infrastructure Engineer",
        "summary": "DevOps Engineer with deep expertise in cloud infrastructure automation, Kubernetes orchestration, CI/CD pipelines, and Python scripting.",
        "skills": ["Kubernetes", "Terraform", "Docker", "Python", "AWS", "CI/CD", "Ansible", "Linux"],
        "experience": [
            {"role": "DevOps Lead", "company": "InfraCloud Global", "years": "2021 - Present",
             "details": "Managed Kubernetes clusters across AWS regions. Automated infrastructure provisioning using Terraform and custom Python scripts."}
        ],
        "education": "B.S. in Information Technology, University of Texas (2019)"
    },
    {
        "filename": "resume_david_miller.pdf",
        "title": "David Miller - Full Stack Engineer",
        "summary": "Versatile Full Stack Engineer experienced in Python web development (Django/Flask) paired with modern React frontends.",
        "skills": ["Python", "Django", "React", "GraphQL", "PostgreSQL", "JavaScript", "Docker"],
        "experience": [
            {"role": "Full Stack Developer", "company": "Nexus Apps Inc.", "years": "2021 - Present",
             "details": "Built end-to-end features for web platform using Django backend and React frontend. Written Python backend APIs and GraphQL schemas."}
        ],
        "education": "B.S. in Computer Science, Georgia Tech (2021)"
    },
    {
        "filename": "resume_eva_davis.pdf",
        "title": "Eva Davis - AI Research & LLM Engineer",
        "summary": "AI Engineer focused on Large Language Models, Retrieval-Augmented Generation (RAG), fine-tuning, and LLM application frameworks in Python.",
        "skills": ["Python", "LLMs", "LangChain", "RAG", "PyTorch", "OpenAI API", "Vector Databases", "Transformers"],
        "experience": [
            {"role": "AI Applications Engineer", "company": "Cognitive AI Systems", "years": "2022 - Present",
             "details": "Implemented enterprise RAG pipelines using Python, LangChain, and vector stores. Integrated OpenAI API and fine-tuned open-source models."}
        ],
        "education": "M.S. in Artificial Intelligence, MIT (2022)"
    }
]

resume_dir = Path("resumes")
resume_dir.mkdir(exist_ok=True)

for r in RESUMES:
    create_pdf_resume(
        filepath=str(resume_dir / r["filename"]),
        title=r["title"],
        summary=r["summary"],
        skills=r["skills"],
        experience=r["experience"],
        education=r["education"]
    )
    print(f"Generated: resumes/{r['filename']}")

print("\nAll sample resumes generated successfully.")

---
## 🛠️ Part A — Core File System Tools

Define the four core tool functions that will be exposed to the LLM. Each returns structured dictionaries for predictable, machine-readable responses.

### Tool 1: `read_file(filepath)` — Read Resume Documents

Reads a file (`.pdf`, `.txt`, `.docx`) and extracts:
- Full text content
- Metadata: name, path, size, type, modification date
- Graceful error handling for missing or unsupported files

In [ ]:
def read_file(filepath: str) -> dict:
    """
    Read a resume file (PDF, TXT, DOCX) and extract text content along with metadata.

    Args:
        filepath (str): Path to the file to read.

    Returns:
        dict: {'status', 'content', 'metadata'} on success,
              {'status', 'error'} on failure.
    """
    try:
        path = Path(filepath)
        if not path.exists():
            return {"status": "failed", "error": f"File not found: {filepath}"}
        if not path.is_file():
            return {"status": "failed", "error": f"Path is not a file: {filepath}"}

        extension = path.suffix.lower()
        content = ""

        if extension == ".txt":
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read()
        elif extension == ".pdf":
            reader = PdfReader(str(path))
            content = "\n".join(
                page.extract_text() for page in reader.pages if page.extract_text()
            )
        elif extension == ".docx":
            doc_obj = docx.Document(str(path))
            content = "\n".join(p.text for p in doc_obj.paragraphs)
        else:
            return {
                "status": "failed",
                "error": f"Unsupported file type: '{extension}'. Supported: .pdf, .txt, .docx"
            }

        stat = path.stat()
        mod_time = datetime.datetime.fromtimestamp(
            stat.st_mtime, tz=datetime.timezone.utc
        ).strftime("%Y-%m-%d %H:%M:%S UTC")

        return {
            "status": "success",
            "content": content,
            "metadata": {
                "name": path.name,
                "filepath": str(path),
                "size_bytes": stat.st_size,
                "type": extension,
                "modified_date": mod_time
            }
        }
    except Exception as e:
        return {"status": "failed", "error": str(e)}


# --- Demo: Read a PDF resume ---
result = read_file("resumes/resume_john_doe.pdf")
print(f"Status  : {result['status']}")
print(f"File    : {result['metadata']['name']}")
print(f"Size    : {result['metadata']['size_bytes']} bytes")
print(f"Modified: {result['metadata']['modified_date']}")
print(f"\nContent Preview (first 300 chars):\n{result['content'][:300]}")

### Tool 2: `list_files(directory, extension)` — List Directory Contents

Lists all files in a directory with:
- Name, path, size, modified date, type
- Optional extension filter (e.g. `.pdf`, `.txt`)

In [ ]:
def list_files(directory: str, extension: str = None) -> list:
    """
    List all files in a directory, optionally filtered by extension.

    Args:
        directory (str): Directory path to list files from.
        extension (str, optional): Extension filter e.g. '.pdf' or 'pdf'.

    Returns:
        list: List of file metadata dicts, or error dict in a list.
    """
    try:
        path = Path(directory)
        if not path.exists() or not path.is_dir():
            return [{"status": "failed", "error": f"Directory not found: {directory}"}]

        ext = None
        if extension:
            ext = extension.strip().lower()
            if not ext.startswith("."):
                ext = f".{ext}"

        files = []
        for item in sorted(path.iterdir()):
            if item.is_file():
                if ext and item.suffix.lower() != ext:
                    continue
                stat = item.stat()
                mod_time = datetime.datetime.fromtimestamp(
                    stat.st_mtime, tz=datetime.timezone.utc
                ).strftime("%Y-%m-%d %H:%M:%S UTC")
                files.append({
                    "name": item.name,
                    "path": str(item),
                    "size_bytes": stat.st_size,
                    "modified_date": mod_time,
                    "type": item.suffix.lower()
                })
        return files
    except Exception as e:
        return [{"status": "failed", "error": str(e)}]


# --- Demo: List all resumes ---
files = list_files("resumes")
print(f"Found {len(files)} resume files:\n")
for f in files:
    print(f"  {f['name']:35s}  {f['size_bytes']:>6} bytes   {f['modified_date']}")

### Tool 3: `write_file(filepath, content)` — Write Files to Disk

Writes text content to any file path:
- Creates parent directories automatically
- Returns success status, path, and file size

In [ ]:
def write_file(filepath: str, content: str) -> dict:
    """
    Write text content to a file, creating parent directories if necessary.

    Args:
        filepath (str): Destination file path.
        content (str): Text content to write.

    Returns:
        dict: {'status', 'message', 'filepath', 'size_bytes'} on success.
    """
    try:
        path = Path(filepath)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        stat = path.stat()
        return {
            "status": "success",
            "message": f"Successfully written to {filepath}",
            "filepath": str(path),
            "size_bytes": stat.st_size
        }
    except Exception as e:
        return {"status": "failed", "error": str(e)}


# --- Demo: Write a summary file ---
sample_content = """RESUME SUMMARY — John Doe
Role   : Senior Software Engineer
Skills : Python, FastAPI, Docker, AWS, PostgreSQL
Status : Strong Python background — recommend for interview.
"""

result = write_file("summaries/john_doe_summary.txt", sample_content)
print(f"Status  : {result['status']}")
print(f"Written : {result['filepath']}")
print(f"Size    : {result['size_bytes']} bytes")

### Tool 4: `search_in_file(filepath, keyword)` — Keyword Search with Context

Searches a document for a keyword (case-insensitive) and returns:
- Line number of each match
- The matched line
- Surrounding context (1 line before and after)

In [ ]:
def search_in_file(filepath: str, keyword: str) -> dict:
    """
    Search for a keyword in a file (case-insensitive) with surrounding context.

    Args:
        filepath (str): File path to search in.
        keyword (str): Keyword or phrase to search for.

    Returns:
        dict: {'status', 'keyword', 'matches_found', 'matches'} on success.
    """
    try:
        read_result = read_file(filepath)
        if read_result.get("status") == "failed":
            return read_result

        content = read_result.get("content", "")
        lines = content.splitlines()
        keyword_lower = keyword.lower()

        matches = []
        for i, line in enumerate(lines):
            if keyword_lower in line.lower():
                start = max(0, i - 1)
                end = min(len(lines), i + 2)
                matches.append({
                    "line_number": i + 1,
                    "match": line.strip(),
                    "context": "\n".join(lines[start:end])
                })

        return {
            "status": "success",
            "filepath": str(filepath),
            "keyword": keyword,
            "matches_found": len(matches),
            "matches": matches
        }
    except Exception as e:
        return {"status": "failed", "error": str(e)}


# --- Demo: Search for 'Python' in John Doe's resume ---
result = search_in_file("resumes/resume_john_doe.pdf", "Python")
print(f"Keyword      : '{result['keyword']}'")
print(f"File         : {result['filepath']}")
print(f"Matches Found: {result['matches_found']}\n")

for m in result["matches"]:
    print(f"  Line {m['line_number']}: {m['match']}")

---
## 🤖 Part B — LLM Function Calling Integration

Define the OpenAI-compatible JSON schemas for each tool and wire them into an LLM-driven agentic loop.
The LLM autonomously decides **which tools to call, in what order, and with what arguments** based on a user query.

### Step 5 — Define Tool JSON Schemas

Each tool is described in an OpenAI function calling schema — this tells the LLM exactly what parameters each tool accepts, so it can invoke them correctly.

In [ ]:
# OpenAI-compatible tool schemas for all 4 file system functions

tools = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read a resume file (.pdf, .txt, .docx) and extract text content along with metadata.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {
                        "type": "string",
                        "description": "The absolute or relative file path to read."
                    }
                },
                "required": ["filepath"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List all files in a directory, optionally filtering by extension (e.g. '.pdf', '.txt', '.docx').",
            "parameters": {
                "type": "object",
                "properties": {
                    "directory": {
                        "type": "string",
                        "description": "The directory path to scan (e.g. 'resumes')."
                    },
                    "extension": {
                        "type": "string",
                        "description": "Optional file extension filter (e.g., '.pdf', '.txt', '.docx')."
                    }
                },
                "required": ["directory"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write text content to a destination file, creating directories if needed.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {
                        "type": "string",
                        "description": "The path of the destination file to write."
                    },
                    "content": {
                        "type": "string",
                        "description": "The textual content to write into the file."
                    }
                },
                "required": ["filepath", "content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_in_file",
            "description": "Perform a case-insensitive keyword search in a file and return matching lines with surrounding context.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filepath": {
                        "type": "string",
                        "description": "The file path to search inside."
                    },
                    "keyword": {
                        "type": "string",
                        "description": "The keyword or phrase to search for."
                    }
                },
                "required": ["filepath", "keyword"]
            }
        }
    }
]

print(f"Registered {len(tools)} tool schemas with the LLM:")
for t in tools:
    params = list(t["function"]["parameters"]["properties"].keys())
    print(f"  - {t['function']['name']}({', '.join(params)})")

### Step 6 — Tool Execution Dispatcher

A dispatcher function maps the LLM's requested function name and arguments to the actual Python implementation.

In [ ]:
def execute_tool_call(tool_call):
    """
    Parse the LLM's tool call request and dispatch to the correct Python function.

    Args:
        tool_call: An OpenAI tool call object with .function.name and .function.arguments.

    Returns:
        str: JSON-serialized result from the tool function.
    """
    function_name = tool_call.function.name
    try:
        arguments = json.loads(tool_call.function.arguments)
    except Exception as e:
        return json.dumps({"status": "failed", "error": f"Invalid arguments: {e}"})

    print(f"  [Tool Call]  {function_name}({json.dumps(arguments)})")

    try:
        if function_name == "read_file":
            result = read_file(arguments.get("filepath"))
        elif function_name == "list_files":
            result = list_files(arguments.get("directory"), arguments.get("extension"))
        elif function_name == "write_file":
            result = write_file(arguments.get("filepath"), arguments.get("content"))
        elif function_name == "search_in_file":
            result = search_in_file(arguments.get("filepath"), arguments.get("keyword"))
        else:
            result = {"status": "failed", "error": f"Unknown tool: {function_name}"}
    except Exception as e:
        result = {"status": "failed", "error": str(e)}

    # Log a short preview of the result
    preview = str(result)
    if len(preview) > 200:
        preview = preview[:200] + " ...[truncated]"
    print(f"  [Tool Result] {preview}")

    return json.dumps(result)


print("Tool execution dispatcher defined.")

### Step 7 — Multi-Step LLM Agentic Loop

The assistant loop:
1. Sends a user query to the LLM with available tool schemas.
2. If the LLM requests tool calls, executes them and appends results to the message history.
3. Repeats until the LLM produces a final natural language response (no more tool calls).

This pattern supports **multi-step** agentic workflows (e.g. `list_files` → `search_in_file` → `write_file`).

In [ ]:
def run_assistant(query: str, max_iterations: int = 10) -> str:
    """
    Run an LLM assistant with multi-step tool calling support.

    The LLM iteratively calls tools until it can formulate a final answer.

    Args:
        query (str): Natural language user query.
        max_iterations (int): Max tool-calling rounds before forcing a stop.

    Returns:
        str: Final response from the LLM.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "You are an AI File System Assistant with tools to read, list, search, and write files. "
                "When asked to perform file operations on resumes or other documents, always use the appropriate tools "
                "to retrieve real data before giving your response. Be thorough, structured, and helpful."
            )
        },
        {"role": "user", "content": query}
    ]

    print("=" * 65)
    print(f"User Query: {query}")
    print("=" * 65)

    for iteration in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

        response_message = response.choices[0].message
        tool_calls = response_message.tool_calls

        if tool_calls:
            # Append LLM tool-call request to message history
            messages.append(response_message)

            # Execute all requested tool calls
            for tool_call in tool_calls:
                tool_result_json = execute_tool_call(tool_call)
                messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": tool_call.function.name,
                    "content": tool_result_json
                })
        else:
            # LLM produced a final answer — no more tool calls needed
            final_answer = response_message.content
            print(f"\nAssistant Response:\n{'=' * 65}")
            print(final_answer)
            print("=" * 65 + "\n")
            return final_answer

    return "[Max tool iterations reached]"


print("LLM assistant loop defined.")

---
## 🧪 Step 8 — Demo Query 1: Read All Resumes

**Query:** *"Read all resumes in the resumes folder"*

**Expected LLM Tool Call Flow:**
1. `list_files(directory="resumes")` → gets list of all resume files
2. `read_file(filepath=...)` → called for each resume
3. LLM summarizes all resumes in a structured response

In [ ]:
response_1 = run_assistant("Read all resumes in the resumes folder")

---
## 🔍 Step 9 — Demo Query 2: Find Resumes Mentioning Python

**Query:** *"Find resumes mentioning Python experience"*

**Expected LLM Tool Call Flow:**
1. `list_files(directory="resumes")` → discover all resume files
2. `search_in_file(filepath=..., keyword="Python")` → for each resume
3. LLM reports which candidates mention Python and where

In [ ]:
response_2 = run_assistant("Find resumes mentioning Python experience")

---
## 📝 Step 10 — Demo Query 3: Create a Summary File

**Query:** *"Create a summary file for resumes/resume_john_doe.pdf at summaries/john_doe_summary.txt"*

**Expected LLM Tool Call Flow:**
1. `read_file(filepath="resumes/resume_john_doe.pdf")` → extract content
2. LLM composes a formatted summary
3. `write_file(filepath="summaries/john_doe_summary.txt", content=...)` → save to disk

In [ ]:
response_3 = run_assistant(
    "Create a summary file for resumes/resume_john_doe.pdf at summaries/john_doe_summary.txt"
)

In [ ]:
# Verify the generated summary file was written to disk
summary_path = Path("summaries/john_doe_summary.txt")
if summary_path.exists():
    print(f"Summary file created: {summary_path}")
    print(f"Size: {summary_path.stat().st_size} bytes\n")
    print("Contents:")
    print("-" * 60)
    print(summary_path.read_text(encoding="utf-8"))
else:
    print("Summary file not found — check if write_file was called.")

---
## ✅ Step 11 — Tool Function Validation Tests

Standalone validation tests verifying each tool function works correctly across all supported file formats.

In [ ]:
def run_tests():
    """Run a suite of validation checks across all four tool functions."""
    passed = 0
    failed = 0

    def check(name, condition, details=""):
        nonlocal passed, failed
        if condition:
            print(f"  PASS  {name}")
            passed += 1
        else:
            print(f"  FAIL  {name}  {details}")
            failed += 1

    print("=" * 55)
    print("Running Tool Validation Tests")
    print("=" * 55)

    # --- list_files tests ---
    print("\n[list_files]")
    files_all = list_files("resumes")
    check("list_files returns a list", isinstance(files_all, list))
    check("list_files finds 5 resumes", len(files_all) == 5, f"got {len(files_all)}")
    check("each result has name + path", all("name" in f and "path" in f for f in files_all))
    pdf_only = list_files("resumes", ".pdf")
    check("extension filter .pdf works", all(f["name"].endswith(".pdf") for f in pdf_only))

    # --- read_file tests ---
    print("\n[read_file]")
    r_pdf = read_file("resumes/resume_john_doe.pdf")
    check("read PDF: status=success", r_pdf["status"] == "success")
    check("read PDF: content contains 'John Doe'", "john doe" in r_pdf.get("content", "").lower())
    check("read PDF: metadata has modified_date", "modified_date" in r_pdf.get("metadata", {}))

    r_miss = read_file("resumes/nonexistent.pdf")
    check("read missing file: status=failed", r_miss["status"] == "failed")

    r_bad = read_file("resumes/resume_john_doe.pdf".replace(".pdf", ".xyz"))
    check("read unsupported type: status=failed", r_bad["status"] == "failed")

    # --- write_file tests ---
    print("\n[write_file]")
    test_content = "Test write content."
    w = write_file("test_output/nested/test.txt", test_content)
    check("write_file: status=success", w["status"] == "success")
    check("write_file: file exists on disk", Path("test_output/nested/test.txt").exists())
    readback = Path("test_output/nested/test.txt").read_text(encoding="utf-8")
    check("write_file: content matches", readback == test_content)

    # --- search_in_file tests ---
    print("\n[search_in_file]")
    s = search_in_file("resumes/resume_john_doe.pdf", "Python")
    check("search: status=success", s["status"] == "success")
    check("search: found matches", s["matches_found"] > 0, f"got {s['matches_found']}")
    check("search: match has line_number", "line_number" in s["matches"][0])
    check("search: match has context", "context" in s["matches"][0])

    s_none = search_in_file("resumes/resume_john_doe.pdf", "ZZZNotAKeywordZZZ")
    check("search: no match returns 0", s_none["matches_found"] == 0)

    # Cleanup test files
    import shutil
    if Path("test_output").exists():
        shutil.rmtree("test_output")

    print(f"\n{'=' * 55}")
    print(f"Results: {passed} passed, {failed} failed")
    print("=" * 55)


run_tests()

---
## 💬 Step 12 — Interactive Mode (Optional)

Run the assistant interactively. Enter any natural language query and watch the LLM invoke tools to answer it. Type `exit` to stop.

In [ ]:
print("Interactive LLM File System Assistant")
print("Type 'exit' to stop.\n")

while True:
    try:
        query = input("Your query: ").strip()
    except (KeyboardInterrupt, EOFError):
        print("\nSession ended.")
        break

    if not query:
        continue
    if query.lower() in ("exit", "quit"):
        print("Session ended.")
        break

    run_assistant(query)